# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import os
from pathlib import Path
import duckdb
import pandas as pd

def load_token():
    for path in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parent.parent / '.env']:
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip().startswith('HF_TOKEN='):
                    return line.split('=', 1)[1].strip().strip(chr(34)).strip(chr(39))
    return os.environ.get('HF_TOKEN')

token = load_token()
if not token:
    raise RuntimeError('HF_TOKEN is required through .env or the environment.')
con = duckdb.connect()
con.execute('CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN ?)', [token])
FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
CONTENT = 'hf://datasets/FlyRank/internship-warehouse/dim_content.parquet'
MIN_IMPRESSIONS = 500
print('HF warehouse connection configured; token value is not displayed.')

HF warehouse connection configured; token value is not displayed.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
query = f"""
WITH daily AS (
  SELECT content_hash_id, client_hash_id,
         SUM(COALESCE(gsc_impressions, 0)) AS impressions,
         SUM(COALESCE(gsc_clicks, 0)) AS clicks,
         SUM(COALESCE(gsc_sum_position, 0)) AS sum_position,
         SUM(COALESCE(ga4_sessions, 0)) AS sessions,
         SUM(COALESCE(ga4_engaged_sessions, 0)) AS engaged_sessions
  FROM read_parquet('{FACT}')
  WHERE gsc_data_available IS TRUE
  GROUP BY content_hash_id, client_hash_id
)
SELECT d.*, c.content_type, c.main_intent, c.word_count,
       d.clicks * 100.0 / NULLIF(d.impressions, 0) AS ctr,
       d.sum_position * 1.0 / NULLIF(d.impressions, 0) AS avg_position
FROM daily d
LEFT JOIN read_parquet('{CONTENT}') c USING (content_hash_id)
WHERE d.impressions >= {MIN_IMPRESSIONS}
"""
queue = con.sql(query).df()
queue['position_tier'] = pd.cut(queue['avg_position'], bins=[0, 3, 10, 20, 50, float('inf')], labels=['top_3', 'page_1', 'striking', 'page_3_5', 'deep'], right=True).astype('string').fillna('no_data')
queue = queue[queue['position_tier'] != 'no_data'].copy()
reference = queue.groupby('position_tier', observed=True)['ctr'].agg(expected_ctr='median', reference_n='size')
queue = queue.join(reference, on='position_tier')
queue['ctr_gap'] = (queue['expected_ctr'] - queue['ctr']).clip(lower=0)
queue['opportunity_score'] = queue['ctr_gap'] * (1 + queue['impressions']).pow(0.5)
queue['below_position_reference'] = queue['ctr'] < queue['expected_ctr']
queue['reason_codes'] = 'sufficient_volume|'
queue.loc[queue['below_position_reference'], 'reason_codes'] += 'below_position_expected_ctr|'
queue.loc[queue['avg_position'] <= 20, 'reason_codes'] += 'visible_position|'
queue.loc[queue['sessions'] >= 30, 'reason_codes'] += 'engagement_context|'
queue['reason_codes'] = queue['reason_codes'].str.rstrip('|')
queue['suggested_action'] = queue['below_position_reference'].map({True: 'review_title_snippet_and_intent', False: 'monitor'})
queue = queue.sort_values(['opportunity_score', 'impressions'], ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1
out_cols = ['rank', 'content_hash_id', 'client_hash_id', 'opportunity_score', 'ctr_gap', 'ctr', 'expected_ctr', 'reference_n', 'position_tier', 'avg_position', 'impressions', 'clicks', 'sessions', 'content_type', 'main_intent', 'reason_codes', 'suggested_action']
output = queue[out_cols]
output_dir = Path('..') / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'baseline_action_score.csv'
output.to_csv(output_path, index=False)
print(f'Wrote {len(output):,} ranked rows to {output_path}')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Wrote 61,924 ranked rows to ..\outputs\baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
top20 = output.head(20).copy()
top20['confidence_note'] = top20.apply(lambda r: 'higher confidence: volume floor and reference n >= 50' if r['reference_n'] >= 50 and r['impressions'] >= 1000 else 'review carefully: borderline evidence', axis=1)
top20['what_would_make_it_wrong'] = 'Position mix, seasonality, or search intent may explain the observed gap.'
print(top20[['rank', 'opportunity_score', 'ctr', 'expected_ctr', 'position_tier', 'impressions', 'reason_codes', 'suggested_action', 'confidence_note', 'what_would_make_it_wrong']].to_string(index=False))

 rank  opportunity_score      ctr  expected_ctr position_tier  impressions                                                                      reason_codes                suggested_action                                       confidence_note                                                  what_would_make_it_wrong
    1         103.531979 0.011299      0.235942         top_3     212404.0 sufficient_volume|below_position_expected_ctr|visible_position|engagement_context review_title_snippet_and_intent higher confidence: volume floor and reference n >= 50 Position mix, seasonality, or search intent may explain the observed gap.
    2          86.413568 0.000741      0.235942         top_3     134984.0                    sufficient_volume|below_position_expected_ctr|visible_position review_title_snippet_and_intent higher confidence: volume floor and reference n >= 50 Position mix, seasonality, or search intent may explain the observed gap.
    3          82.825255 0.000806      0.235942  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
weak = output[(output['rank'] <= 20) & ((output['reference_n'] < 50) | (output['impressions'] < 1000))]
print(f'Potential weak picks in top 20: {len(weak)}')
print(weak[['rank', 'impressions', 'reference_n', 'ctr', 'expected_ctr', 'position_tier']].to_string(index=False))
forbidden = {'health_score', 'priority_score', 'action_type', 'trend_direction', 'trend_pct', 'url', 'query', 'title'}
feature_names = {'impressions', 'clicks', 'sum_position', 'sessions', 'engaged_sessions', 'ctr', 'avg_position', 'content_type', 'main_intent', 'word_count'}
assert not (feature_names & forbidden)
assert output['rank'].is_unique and output['rank'].min() == 1
assert output['impressions'].ge(MIN_IMPRESSIONS).all()
assert output['content_hash_id'].notna().all()
print('Leakage, ranking, volume, and public-safe output checks passed.')

Potential weak picks in top 20: 0
Empty DataFrame
Columns: [rank, impressions, reference_n, ctr, expected_ctr, position_tier]
Index: []
Leakage, ranking, volume, and public-safe output checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.